In [1]:
import requests

URL = "https://books.toscrape.com/catalogue/page-1.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (ton.email@exemple.fr)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"            # <- a ne pas oublier, on voit pourquoi juste apres

print("status   :", r.status_code)
print("longueur :", len(r.text), "caracteres")
print("debut    :", r.text[:60])

status   : 200
longueur : 50449 caracteres
debut    : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [2]:
r = requests.get(URL, headers=ENTETES, timeout=10)

print("encodage devine par requests :", r.encoding)
print("encodage reel de la page     :", r.apparent_encoding)
print("le prix :", r.text[r.text.find("price_color") : r.text.find("price_color") + 40])

encodage devine par requests : ISO-8859-1
encodage reel de la page     : utf-8
le prix : price_color">Â£51.77</p>
    

<p class=


In [3]:
print("le prix est present :", "£51.77" in r.text)
print("le titre est present:", "A Light in the Attic" in r.text)

le prix est present : True
le titre est present: True


In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

fiches = soup.select("article.product_pod")
print("fiches trouvees :", len(fiches))

fiches trouvees : 20


In [5]:
fiche = fiches[0]

# le CONTENU d'une balise
prix = fiche.select_one("p.price_color").get_text(strip=True)

# un ATTRIBUT
lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]
href = lien_h3["href"]

print("prix  :", prix)
print("titre :", titre)
print("href  :", href)

prix  : Â£51.77
titre : A Light in the Attic
href  : a-light-in-the-attic_1000/index.html


In [6]:
classes = fiche.select_one("p.star-rating")["class"]
print("classes :", classes)

classes : ['star-rating', 'Three']


In [7]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

note = NOTES[classes[1]]
print("note :", note)

note : 3


In [8]:
from urllib.parse import urljoin

print(urljoin(URL, href))

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [9]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }


print(lire_fiche(fiches[0], URL))

{'titre': 'A Light in the Attic', 'prix': 'Â£51.77', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [10]:
suivant = soup.select_one("li.next a")
print(suivant["href"] if suivant else "pas de page suivante")

page-2.html


In [11]:
import time

def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres = []
    page = 0

    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"  page {page} : status {r.status_code}, on s'arrete")
            break

        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"  page {page} : {len(fiches)} livres  (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)

    return livres


livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/page-1.html")
print("collecte :", len(livres), "livres")

  page 1 : 20 livres  (total 20)
  page 2 : 20 livres  (total 40)
  page 3 : 20 livres  (total 60)
collecte : 60 livres


In [12]:
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None}

    soup = BeautifulSoup(r.text, "html.parser")

    # le tableau : une entree par ligne, le th en cle et le td en valeur
    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)

    # la categorie : troisieme element du fil d'Ariane
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }


url_test = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
print(lire_detail(url_test))

{'categorie': 'Poetry', 'stock': 'In stock (22 available)', 'upc': 'a897fe39b1053632'}


In [13]:
import re

def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0


print(nombre_en_stock("In stock (22 available)"))
print(nombre_en_stock("Out of stock"))
print(nombre_en_stock(None))

22
0
None


# Fiche technique — Catégorie Poetry

**Site :** Books to Scrape  
**Catégorie :** Poetry  
**Nombre de livres annoncé :** 19  
**Nombre de pages :** 1  
**Sélecteur du bloc répété :** `article.product_pod`  
**Sélecteur de la page suivante :** `li.next a` — absent dans cette catégorie.

## Tableau des données

| Donnée | Sélecteur CSS | Contenu ou attribut ? |
|---|---|---|
| Titre | `h3 a` | Attribut `title` |
| Prix | `p.price_color` | Contenu |
| Disponibilité | `p.instock` | Contenu |
| Note | `p.star-rating` | Attribut `class` |
| Lien vers la fiche | `h3 a` | Attribut `title` |

**Objectif :** collecter les 19 livres de la catégorie Poetry, vérifier que toutes les informations sont présentes et arrêter la pagination puisqu'il n'existe pas de deuxième page.

In [14]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://books.toscrape.com/catalogue/category/books/poetry_23/index.html"

# Télécharger la page Poetry
r = requests.get(URL, timeout=10)
r.raise_for_status()

# Analyser son HTML
soup = BeautifulSoup(r.text, "html.parser")

# Récupérer les fiches de Poetry
fiches = soup.select("article.product_pod")

# Collecter les informations
livres_poetry = []

for fiche in fiches:
    infos = lire_fiche(fiche, URL)
    livres_poetry.append(infos)

# Créer le DataFrame
df = pd.DataFrame(livres_poetry)

display(df)

,titre,prix,dispo,note,url
0,A Light in the Attic,Â£51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,The Black Maria,Â£52.15,In stock,1,https://books.toscrape.com/catalogue/the-black...
2,Shakespeare's Sonnets,Â£20.66,In stock,4,https://books.toscrape.com/catalogue/shakespea...
3,Olio,Â£23.88,In stock,1,https://books.toscrape.com/catalogue/olio_984/...
4,You can't bury them all: Poems,Â£33.63,In stock,2,https://books.toscrape.com/catalogue/you-cant-...
5,Slow States of Collapse: Poems,Â£57.31,In stock,3,https://books.toscrape.com/catalogue/slow-stat...
6,Untitled Collection: Sabbath Poems 2014,Â£14.27,In stock,4,https://books.toscrape.com/catalogue/untitled-...
7,Poems That Make Grown Women Cry,Â£14.19,In stock,4,https://books.toscrape.com/catalogue/poems-tha...
8,Night Sky with Exit Wounds,Â£41.05,In stock,1,https://books.toscrape.com/catalogue/night-sky...
9,salt.,Â£46.78,In stock,4,https://books.toscrape.com/catalogue/salt_731/...


In [15]:
URL = "https://books.toscrape.com/catalogue/category/books/poetry_23/index.html"

print("Nombre total de livres :", len(livres_poetry))

Nombre total de livres : 19


In [16]:
URL_POETRY = "https://books.toscrape.com/catalogue/category/books/poetry_23/index.html"

livres_poetry = collecter_toutes_les_pages(URL_POETRY, pages_max=3)

print("Nombre total de livres :", len(livres_poetry))

  page 1 : 19 livres  (total 19)
Nombre total de livres : 19


In [17]:
import time

for i, livre in enumerate(livres_poetry, start=1):

    detail = lire_detail(livre["url"])

    livre["categorie"] = detail["categorie"]
    livre["stock"] = nombre_en_stock(detail["stock"])
    livre["upc"] = detail["upc"]

    print(f"Livre {i}/{len(livres_poetry)} : {livre['titre']}")

    time.sleep(1)

Livre 1/19 : A Light in the Attic
Livre 2/19 : The Black Maria
Livre 3/19 : Shakespeare's Sonnets
Livre 4/19 : Olio
Livre 5/19 : You can't bury them all: Poems
Livre 6/19 : Slow States of Collapse: Poems
Livre 7/19 : Untitled Collection: Sabbath Poems 2014
Livre 8/19 : Poems That Make Grown Women Cry
Livre 9/19 : Night Sky with Exit Wounds
Livre 10/19 : salt.
Livre 11/19 : Quarter Life Poetry: Poems for the Young, Broke and Hangry
Livre 12/19 : Out of Print: City Lights Spotlight No. 14
Livre 13/19 : Les Fleurs du Mal
Livre 14/19 : Howl and Other Poems
Livre 15/19 : Leave This Song Behind: Teen Poetry at Its Best
Livre 16/19 : The Collected Poems of W.B. Yeats (The Collected Works of W.B. Yeats #1)
Livre 17/19 : The Crossover
Livre 18/19 : Booked
Livre 19/19 : Twenty Love Poems and a Song of Despair


In [18]:
df = pd.DataFrame(livres_poetry)
display(df)

,titre,prix,dispo,note,url,categorie,stock,upc
0,A Light in the Attic,£51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...,Poetry,22,a897fe39b1053632
1,The Black Maria,£52.15,In stock,1,https://books.toscrape.com/catalogue/the-black...,Poetry,19,1dfe412b8ac00530
2,Shakespeare's Sonnets,£20.66,In stock,4,https://books.toscrape.com/catalogue/shakespea...,Poetry,19,30a7f60cd76ca58c
3,Olio,£23.88,In stock,1,https://books.toscrape.com/catalogue/olio_984/...,Poetry,19,feb7cc7701ecf901
4,You can't bury them all: Poems,£33.63,In stock,2,https://books.toscrape.com/catalogue/you-cant-...,Poetry,17,55f9da0c5eea2e10
5,Slow States of Collapse: Poems,£57.31,In stock,3,https://books.toscrape.com/catalogue/slow-stat...,Poetry,17,b4fd5943413e089a
6,Untitled Collection: Sabbath Poems 2014,£14.27,In stock,4,https://books.toscrape.com/catalogue/untitled-...,Poetry,16,657fe5ead67a7767
7,Poems That Make Grown Women Cry,£14.19,In stock,4,https://books.toscrape.com/catalogue/poems-tha...,Poetry,15,31001a1210623bd4
8,Night Sky with Exit Wounds,£41.05,In stock,1,https://books.toscrape.com/catalogue/night-sky...,Poetry,15,cfeb1e1e0e695b58
9,salt.,£46.78,In stock,4,https://books.toscrape.com/catalogue/salt_731/...,Poetry,14,86cbddb61ea78bb7


In [19]:
detail = lire_detail(livres_poetry[0]["url"])

print("Stock brut :", detail["stock"])
print("Stock chiffré :", nombre_en_stock(detail["stock"]))

Stock brut : In stock (22 available)
Stock chiffré : 22


In [ ]:
import time

for i, livre in enumerate(livres_poetry, start=1):

    detail = lire_detail(livre["url"])

    livre["categorie"] = detail["categorie"]
    livre["stock"] = nombre_en_stock(detail["stock"])
    livre["upc"] = detail["upc"]

    print(f"Livre {i}/{len(livres_poetry)} : stock = {livre['stock']}")

    if i < len(livres_poetry):
        time.sleep(1)

Livre 1/19 : stock = 22
Livre 2/19 : stock = 19
Livre 3/19 : stock = 19
Livre 4/19 : stock = 19
Livre 5/19 : stock = 17
Livre 6/19 : stock = 17
Livre 7/19 : stock = 16
Livre 8/19 : stock = 15
Livre 9/19 : stock = 15
Livre 10/19 : stock = 14
Livre 11/19 : stock = 14
Livre 12/19 : stock = 8
Livre 13/19 : stock = 8
Livre 14/19 : stock = 8
Livre 15/19 : stock = 7


In [ ]:
df = pd.DataFrame(livres_poetry)

display(df[["titre", "categorie", "stock", "upc"]])

,titre,categorie,stock,upc
0,A Light in the Attic,Poetry,22,a897fe39b1053632
1,The Black Maria,Poetry,19,1dfe412b8ac00530
2,Shakespeare's Sonnets,Poetry,19,30a7f60cd76ca58c
3,Olio,Poetry,19,feb7cc7701ecf901
4,You can't bury them all: Poems,Poetry,17,55f9da0c5eea2e10
5,Slow States of Collapse: Poems,Poetry,17,b4fd5943413e089a
6,Untitled Collection: Sabbath Poems 2014,Poetry,16,657fe5ead67a7767
7,Poems That Make Grown Women Cry,Poetry,15,31001a1210623bd4
8,Night Sky with Exit Wounds,Poetry,15,cfeb1e1e0e695b58
9,salt.,Poetry,14,86cbddb61ea78bb7


In [ ]:
print("Stock brut :", detail["stock"])
print("Stock chiffré :", nombre_en_stock(detail["stock"]))

Stock brut : In stock (1 available)
Stock chiffré : 1


In [ ]:
import pandas as pd

df = pd.DataFrame(livres_poetry)

df['stock'] = df['stock'].astype(int)

display(df)

,titre,prix,dispo,note,url,categorie,stock,upc
0,A Light in the Attic,£51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...,Poetry,22,a897fe39b1053632
1,The Black Maria,£52.15,In stock,1,https://books.toscrape.com/catalogue/the-black...,Poetry,19,1dfe412b8ac00530
2,Shakespeare's Sonnets,£20.66,In stock,4,https://books.toscrape.com/catalogue/shakespea...,Poetry,19,30a7f60cd76ca58c
3,Olio,£23.88,In stock,1,https://books.toscrape.com/catalogue/olio_984/...,Poetry,19,feb7cc7701ecf901
4,You can't bury them all: Poems,£33.63,In stock,2,https://books.toscrape.com/catalogue/you-cant-...,Poetry,17,55f9da0c5eea2e10
5,Slow States of Collapse: Poems,£57.31,In stock,3,https://books.toscrape.com/catalogue/slow-stat...,Poetry,17,b4fd5943413e089a
6,Untitled Collection: Sabbath Poems 2014,£14.27,In stock,4,https://books.toscrape.com/catalogue/untitled-...,Poetry,16,657fe5ead67a7767
7,Poems That Make Grown Women Cry,£14.19,In stock,4,https://books.toscrape.com/catalogue/poems-tha...,Poetry,15,31001a1210623bd4
8,Night Sky with Exit Wounds,£41.05,In stock,1,https://books.toscrape.com/catalogue/night-sky...,Poetry,15,cfeb1e1e0e695b58
9,salt.,£46.78,In stock,4,https://books.toscrape.com/catalogue/salt_731/...,Poetry,14,86cbddb61ea78bb7


In [ ]:
# combien de livres, et quel stock total ?

print(len(livres_poetry))
total_stock = df['stock'].sum()
print(total_stock)
df['prix'] = df['prix'].str.replace('£', '', regex=False)


19
235


AttributeError: Can only use .str accessor with string values, not floating

In [ ]:
# quel est le prix moyen, et le livre le plus cher ?

print(round(df['prix'].mean(), 2))
print(df['prix'].max())

35.97
57.31
